# Tracing `get_grad_embedding` (BADGE last-layer gradient embedding)

This notebook builds the **same two models the experiments build** and then runs
`src/query/query_diversity.py::get_grad_embedding` on each, printing the shape of
every important object (`features`, `outputs`, `targets`, `l0_grads`,
`l1_grads`, ...).

Two examples:

| | CIFAR-10 | MIMIC phenotyping |
|---|---|---|
| model cfg | `resnet` (`name: bayesian_resnet`, `small_head: True`) | `sand` (`name: sand`, `small_head: False`) |
| data cfg | `cifar10` (`num_classes=10`, `shape=[32,32,3]`) | `mimic3_pheno` (`num_classes=25`, `shape=[24,387]`, `multilabel: true`) |
| feature dim `Z` | 512 (resnet18 backbone) | 832 (`hid_dim*M + hid_dim`) |
| head | `Sequential(Dropout, Linear(512,10))` | plain `Linear(832,25)` |
| task | single-label softmax | multi-label sigmoid/BCE |

> **Kernel:** run this notebook with the `real-al` conda environment
> (`conda activate real-al`, then launch Jupyter, or pick the `real-al` kernel).
> Everything below runs on **synthetic** batches by default, so no dataset files
> or downloads are needed — flip `USE_REAL_DATA` in the data section to use the
> real pool loader.

The full function implementation is pasted verbatim in the
**"The function, copied verbatim"** section so you can edit and re-run it.


## 1. Environment setup

In [1]:
import os, sys

PROJECT = "/home/jing/Desktop/realistic-al"
SRC = os.path.join(PROJECT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)                 # so `import models...`, `import data...` work

# Load DATA_ROOT / EXPERIMENT_ROOT from .env (cfg interpolations reference them).
with open(os.path.join(PROJECT, ".env")) as fh:
    for line in fh:
        line = line.strip()
        if line.startswith("export "):
            k, _, v = line[len("export "):].partition("=")
            os.environ.setdefault(k.strip(), v.strip())
print("DATA_ROOT       =", os.environ.get("DATA_ROOT"))
print("EXPERIMENT_ROOT =", os.environ.get("EXPERIMENT_ROOT"))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

DATA_ROOT       = /home/jing/Desktop/realistic-al/datasets
EXPERIMENT_ROOT = /home/jing/Desktop/realistic-al/experiments
torch 2.9.1+cu128 | device: cuda:0


## 2. Compose the config exactly like a run

Runs are launched through Hydra with `config_path="./config", config_name="config"`
(see `src/main.py`). A launcher just passes group/param **overrides**
(e.g. `model=resnet data=cifar10 ...`). We reproduce that with Hydra's
`compose` API instead of the CLI.

In [2]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

CONFIG_DIR = os.path.join(SRC, "config")

def compose_cfg(overrides):
    """Same DictConfig Hydra builds for a run: config.yaml + overrides."""
    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=CONFIG_DIR, version_base="1.1"):
        return compose(config_name="config", overrides=overrides)

In [3]:
# Overrides mirror the experiment launchers:
#   CIFAR: launchers/exp_cifar10imb_basic_tune_kernel.py  (resnet, small_head, dropout 0.5)
#   PHENO: launchers/sweep_exp_mimic3_pheno_basic.py       (sand, multilabel, dropout 0.2)
# active/optim/query are set for authenticity but are irrelevant to get_grad_embedding.
cifar_overrides = [
    "model=resnet",          # name: bayesian_resnet, small_head: True
    "data=cifar10",          # num_classes=10, shape=[32,32,3]
    "active=cifar10_med",
    "optim=sgd_cosine",
    "query=badge",
    "model.dropout_p=0.5",
    "trainer.seed=12345",
]
pheno_overrides = [
    "model=sand",            # name: sand, small_head: False
    "data=mimic3_pheno",     # num_classes=25, shape=[24,387], multilabel: true
    "active=mimic3_med",
    "optim=adam",
    "query=badge",
    "model.dropout_p=0.2",
    "model.weighted_loss=True",
    "trainer.seed=12345",
]

cfg_cifar = compose_cfg(cifar_overrides)
cfg_pheno = compose_cfg(pheno_overrides)

for name, cfg in [("CIFAR-10", cfg_cifar), ("MIMIC-pheno", cfg_pheno)]:
    print(f"=== {name} ===")
    print("  model.name       :", cfg.model.name)
    print("  model.small_head :", cfg.model.small_head)
    print("  model.dropout_p  :", cfg.model.dropout_p)
    print("  data.num_classes :", cfg.data.num_classes)
    print("  data.shape       :", OmegaConf.to_container(cfg.data.shape))
    print("  data.multilabel  :", cfg.data.get("multilabel", False))
    print()

=== CIFAR-10 ===
  model.name       : bayesian_resnet
  model.small_head : True
  model.dropout_p  : 0.5
  data.num_classes : 10
  data.shape       : [32, 32, 3]
  data.multilabel  : False

=== MIMIC-pheno ===
  model.name       : sand
  model.small_head : False
  model.dropout_p  : 0.2
  data.num_classes : 25
  data.shape       : [24, 387]
  data.multilabel  : True



## 3. Build the models (exactly as the trainer does)

`ActiveTrainingLoop._init_model` is literally `self.model = BayesianModule(self.cfg)`
(`src/trainer.py`). `BayesianModule.__init__` reads `cfg.data` / `cfg.model`, calls
`build_model` (registry lookup on `cfg.model.name`), and sets `self.multilabel`.
No dataset files are touched — `resnet18` is built with random init (`weights=None`).

In [4]:
from models.bayesian import BayesianModule

def build_bayesian_module(cfg):
    torch.manual_seed(cfg.trainer.seed)
    model = BayesianModule(cfg)     # == trainer._init_model
    return model.eval().to(DEVICE)

model_cifar = build_bayesian_module(cfg_cifar)
model_pheno = build_bayesian_module(cfg_pheno)

for name, m in [("CIFAR-10", model_cifar), ("MIMIC-pheno", model_pheno)]:
    print(f"=== {name} ===")
    print("  wrapper            :", type(m).__name__)
    print("  backbone (m.model) :", type(m.model).__name__)
    print("  m.multilabel       :", m.multilabel)
    print("  hparams.small_head :", m.hparams.model.small_head)
    print("  classifier is Seq  :", isinstance(m.model.classifier, nn.Sequential))
    print("  classifier         :", m.model.classifier)
    print()

/home/jing/anaconda3/envs/real-al/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/home/jing/anaconda3/envs/real-al/lib/python3.10/site-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
  "lr_options": generate_power_seq(LEARNING_RATE_CIFAR, 11),
/home/jing/anaconda3/envs/real-al/lib/python3.10/site-packages/pl_bolts/models/self_supe

=== CIFAR-10 ===
  wrapper            : BayesianModule
  backbone (m.model) : ResNet
  m.multilabel       : False
  hparams.small_head : True
  classifier is Seq  : True
  classifier         : Sequential(
  (0): ConsistentMCDropout()
  (1): Linear(in_features=512, out_features=10, bias=True)
)

=== MIMIC-pheno ===
  wrapper            : BayesianModule
  backbone (m.model) : BayesianSANDModel
  m.multilabel       : True
  hparams.small_head : False
  classifier is Seq  : False
  classifier         : Linear(in_features=832, out_features=25, bias=True)



## 4. A pool dataloader for each model

`get_grad_embedding` iterates `for (x, y) in dataloader`. The two datasets have
different `(x, y)` shapes, which is exactly why the input-handling branch at the
top of the function exists:

* **CIFAR-10** — `x` is an image tensor `(3,32,32)`, `y` an int class label.
* **MIMIC-pheno** — `x` is a **tuple** `(ts (24,387), demo (2,))`, `y` a `(25,)` float multi-hot
  (see `MIMIC3PhenotypeDataset.__getitem__`).

Default = synthetic batches (guaranteed to run). Set `USE_REAL_DATA = True` to use
the real `datamodule.pool_dataloader()` instead (needs CIFAR-10 / MIMIC `.npy`
files under `$DATA_ROOT`).

In [5]:
POOL_N, BATCH = 16, 8   # small so the per-batch prints stay readable

class SyntheticCIFARPool(Dataset):
    def __init__(self, n, shape, num_classes, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.x = torch.randn(n, *shape, generator=g)          # (n,3,32,32)
        self.y = torch.randint(0, num_classes, (n,), generator=g)
    def __len__(self):  return len(self.x)
    def __getitem__(self, i):  return self.x[i], int(self.y[i])

class SyntheticPhenoPool(Dataset):
    def __init__(self, n, T, threeV, d_static, num_classes, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.ts   = torch.randn(n, T, threeV, generator=g)    # (n,24,387)
        self.demo = torch.randn(n, d_static, generator=g)     # (n,2)
        self.y    = (torch.rand(n, num_classes, generator=g) > 0.7).float()  # (n,25)
    def __len__(self):  return len(self.ts)
    def __getitem__(self, i):  return (self.ts[i], self.demo[i]), self.y[i]

USE_REAL_DATA = False

if not USE_REAL_DATA:
    # NB shape is [H,W,C] for CIFAR in the cfg; the loader yields (C,H,W).
    loader_cifar = DataLoader(SyntheticCIFARPool(POOL_N, (3, 32, 32), 10), batch_size=BATCH)
    loader_pheno = DataLoader(SyntheticPhenoPool(POOL_N, 24, 387, 2, 25), batch_size=BATCH)
else:
    # Real pool loader, exactly as QuerySampler builds it (src/query/query.py:69).
    from run_training import get_torchvision_dm, label_active_dm
    def real_pool_loader(cfg):
        dm = get_torchvision_dm(cfg, active_dataset=True)
        label_active_dm(cfg, cfg.active.num_labelled, cfg.active.balanced, dm)
        return dm.pool_dataloader(batch_size=BATCH)
    loader_cifar = real_pool_loader(cfg_cifar)
    loader_pheno = real_pool_loader(cfg_pheno)

# peek at one batch of each
(xc, yc) = next(iter(loader_cifar))
print("CIFAR  x:", tuple(xc.shape), "| y:", tuple(yc.shape), yc.dtype)
(xp, yp) = next(iter(loader_pheno))
print("PHENO  x: (ts", tuple(xp[0].shape), ", demo", tuple(xp[1].shape), ") | y:", tuple(yp.shape), yp.dtype)

CIFAR  x: (8, 3, 32, 32) | y: (8,) torch.int64
PHENO  x: (ts (8, 24, 387) , demo (8, 2) ) | y: (8, 25) torch.float32


## 5. The function, copied verbatim

Pasted unchanged from `src/query/query_diversity.py` (the `disable_mc_dropout`
helper it depends on, then `get_grad_embedding`). Edit this cell freely and
re-run the calls below — it shadows the imported version.

In [6]:
from contextlib import contextmanager
from models.bayesian_module import BayesianModule, ConsistentMCDropout


@contextmanager
def disable_mc_dropout(model: torch.nn.Module):
    """Temporarily neutralize every ConsistentMCDropout in ``model``.

    ConsistentMCDropout keeps applying dropout even in eval mode (MC-dropout by
    design, see ``models.bayesian_module``). That is what we want for BALD/uncertainty
    acquisition, but NOT for the BADGE-style gradient embeddings computed here:
    the head's dropout randomizes the softmax output (hence the error vector
    ``p_x - e_y``, and even the argmax pseudo-label) and, for non-small-head
    models, zeros out ~p of the penultimate-feature entries that form the
    outer-product embedding. Both make the last-layer gradient stochastic, whereas
    BADGE/gvendi assume a deterministic embedding.

    We set ``p=0`` and drop the cached mask for the duration of the ``with`` block,
    then restore the original ``p`` (and reset the mask so the next real forward
    rebuilds it at the restored ``p``). The uncertainty/BALD path, which runs its
    own forward passes outside this context, is unaffected.
    """
    touched = []
    for module in model.modules():
        if isinstance(module, ConsistentMCDropout):
            touched.append((module, module.p))
            module.p = 0.0
            module.reset()  # discard any mask cached at the old p
    try:
        yield
    finally:
        for module, p in touched:
            module.p = p
            module.reset()  # force a fresh mask at the restored p on next use


def get_grad_embedding(
    model: torch.nn.Module,
    dataloader: DataLoader,
    grad_embedding_type: str = "linear",
    device: str = "cuda:0",
    use_true_labels: bool = False,
) -> torch.Tensor:
    """Compute gradient embedding of final layer of `model` for each input in `dataloader`.

    The per-sample gradient of the cross-entropy loss w.r.t. the last linear layer
    factorizes as g_x = (p_x - e_{y_x}) h_x^T, where p_x is the softmax output,
    e_{y_x} the one-hot label and h_x the penultimate features. "bias" returns only
    the error vector (p_x - e_{y_x}) (C-dim), "linear" the flattened outer product
    (C*Z-dim), anything else the concatenation of both.
    """

    BayesianModule.k = 1
    start_index = 0
    assert hasattr(model, "model")  # model requires sub class model (due to code)
    assert hasattr(
        model.model, "classifier"
    )  # model.model requires classification head (due to code).
    with disable_mc_dropout(model):
        for i, (x, y) in enumerate(dataloader):
            if isinstance(x, (list, tuple)):
                inputs = [t.to(device) if isinstance(t, torch.Tensor) else t for t in x]
            else:
                inputs = x.to(device)
            with torch.no_grad():
                features = model.get_features(inputs)  # B x Z
            l1 = features  # B x Z
            if not model.hparams.model.small_head and isinstance(model.model.classifier, nn.Sequential):
                l1 = model.model.classifier[:-1](features)
            embDim = l1.shape[-1]
            outputs = model.model.classifier(features)

            if getattr(model, "multilabel", False):
                if use_true_labels:
                    targets = y.to(device).float()  # (B, C) multi-hot
                else:
                    targets = (torch.sigmoid(outputs) > 0.5).float()
                loss = F.binary_cross_entropy_with_logits(outputs, targets, reduction="sum")
            else:
                if use_true_labels:
                    targets = y.to(device).long()
                else:
                    targets = torch.argmax(outputs, dim=1)
                loss = F.cross_entropy(outputs, targets, reduction="sum")
            l0_grads = torch.autograd.grad(loss, outputs)[0]  # B x C

            if grad_embedding_type != "bias":
                l0_expand = torch.repeat_interleave(l0_grads, embDim, dim=1)
                l1_grads = l0_expand * l1.repeat(1, outputs.shape[-1])  # B x C*Z

            if grad_embedding_type == "bias":
                gradient_embedding = l0_grads
            elif grad_embedding_type == "linear":
                gradient_embedding = l1_grads
            else:
                gradient_embedding = torch.cat([l0_grads, l1_grads], dim=1)
            if i == 0:
                gradient_embeddings = torch.empty(
                    [len(dataloader.dataset), gradient_embedding.shape[1]], device="cpu"
                )
            gradient_embeddings[
                start_index : start_index + gradient_embedding.shape[0]
            ] = gradient_embedding.detach().to("cpu")
            start_index += gradient_embedding.shape[0]
            torch.cuda.empty_cache()
    return gradient_embeddings

In [8]:
# Sanity: the verbatim copy returns the expected (N, C*Z) shape for each model.
emb_c = get_grad_embedding(model_cifar, loader_cifar, device=DEVICE)
emb_p = get_grad_embedding(model_pheno, loader_pheno, device=DEVICE)
print("CIFAR-10   embedding:", tuple(emb_c.shape), "  (expect (16, 10*512=5120))")
print("MIMIC-pheno embedding:", tuple(emb_p.shape), "  (expect (16, 25*832=20800))")

CIFAR-10   embedding: (16, 5120)   (expect (16, 10*512=5120))
MIMIC-pheno embedding: (16, 20800)   (expect (16, 25*832=20800))


In [10]:
dataloader, model = loader_cifar, model_cifar
device=DEVICE
grad_embedding_type = "linear"
device = "cuda:0"
use_true_labels = False
i, (x, y) = next(enumerate(dataloader))

In [11]:
if isinstance(x, (list, tuple)):
    inputs = [t.to(device) if isinstance(t, torch.Tensor) else t for t in x]
else:
    inputs = x.to(device)

In [12]:
inputs.shape

torch.Size([8, 3, 32, 32])

In [13]:
with torch.no_grad():
    features = model.get_features(inputs)  # B x Z
l1 = features  # B x Z
if not model.hparams.model.small_head and isinstance(model.model.classifier, nn.Sequential):
    l1 = model.model.classifier[:-1](features)
embDim = l1.shape[-1]
outputs = model.model.classifier(features)

In [23]:
if getattr(model, "multilabel", False):
    if use_true_labels:
        targets = y.to(device).float()  # (B, C) multi-hot
    else:
        targets = (torch.sigmoid(outputs) > 0.5).float()
    loss = F.binary_cross_entropy_with_logits(outputs, targets, reduction="sum")
else:
    if use_true_labels:
        targets = y.to(device).long()
    else:
        targets = torch.argmax(outputs, dim=1)
    loss = F.cross_entropy(outputs, targets, reduction="sum")
l0_grads = torch.autograd.grad(loss, outputs)

In [25]:
l0_grads

(tensor([[ 0.0274,  0.0437,  0.0280,  0.0130,  0.0219,  0.3366,  0.0148,  0.0248,
           0.0037, -0.5140],
         [ 0.0247,  0.0425,  0.0352,  0.0134,  0.0256,  0.3436,  0.0174,  0.0262,
           0.0040, -0.5326],
         [ 0.0281,  0.0470,  0.0296,  0.0124,  0.0270,  0.3177,  0.0137,  0.0212,
           0.0043, -0.5012],
         [ 0.0256,  0.0465,  0.0288,  0.0141,  0.0294,  0.3515,  0.0149,  0.0281,
           0.0045, -0.5435],
         [ 0.0249,  0.0480,  0.0263,  0.0141,  0.0265,  0.3278,  0.0165,  0.0245,
           0.0042, -0.5129],
         [ 0.0218,  0.0428,  0.0298,  0.0110,  0.0228,  0.3556,  0.0136,  0.0212,
           0.0034, -0.5220],
         [ 0.0279,  0.0505,  0.0318,  0.0140,  0.0292,  0.3250,  0.0156,  0.0281,
           0.0045, -0.5265],
         [ 0.0244,  0.0463,  0.0253,  0.0133,  0.0267,  0.3004,  0.0128,  0.0213,
           0.0039, -0.4744]], device='cuda:0'),)

In [21]:
loss

tensor(5.8100, device='cuda:0', grad_fn=<NllLossBackward0>)

In [20]:
l0_grads.shape

torch.Size([8, 10])

In [ ]:

if grad_embedding_type != "bias":
    l0_expand = torch.repeat_interleave(l0_grads, embDim, dim=1)
    l1_grads = l0_expand * l1.repeat(1, outputs.shape[-1])  # B x C*Z

if grad_embedding_type == "bias":
    gradient_embedding = l0_grads
elif grad_embedding_type == "linear":
    gradient_embedding = l1_grads
else:
    gradient_embedding = torch.cat([l0_grads, l1_grads], dim=1)
if i == 0:
    gradient_embeddings = torch.empty(
        [len(dataloader.dataset), gradient_embedding.shape[1]], device="cpu"
    )
gradient_embeddings[
    start_index : start_index + gradient_embedding.shape[0]
] = gradient_embedding.detach().to("cpu")
start_index += gradient_embedding.shape[0]

## 6. Instrumented trace

Same body as above, with `print`s that dump the shape/dtype of every intermediate
for the **first batch**. This is the copy to read alongside the shapes discussion.

In [ ]:
def get_grad_embedding_traced(
    model, dataloader, grad_embedding_type="linear", device=DEVICE, use_true_labels=False, tag=""
):
    print(f"################  {tag}  (grad_embedding_type={grad_embedding_type!r}, "
          f"use_true_labels={use_true_labels})")
    BayesianModule.k = 1
    start_index = 0
    with disable_mc_dropout(model):
        for i, (x, y) in enumerate(dataloader):
            trace = (i == 0)                       # only narrate the first batch
            # ---- input handling ----
            if isinstance(x, (list, tuple)):
                inputs = [t.to(device) if isinstance(t, torch.Tensor) else t for t in x]
                if trace: print("  x is a", type(x).__name__, "->",
                                [tuple(t.shape) for t in inputs])
            else:
                inputs = x.to(device)
                if trace: print("  x is a Tensor ->", tuple(inputs.shape))
            if trace: print("  y:", tuple(y.shape), y.dtype)

            with torch.no_grad():
                features = model.get_features(inputs)           # B x Z
            l1 = features
            branch = (not model.hparams.model.small_head
                      and isinstance(model.model.classifier, nn.Sequential))
            if branch:
                l1 = model.model.classifier[:-1](features)
            embDim = l1.shape[-1]
            outputs = model.model.classifier(features)          # B x C
            if trace:
                print(f"  features (B x Z)      : {tuple(features.shape)}")
                print(f"  small_head={model.hparams.model.small_head}  "
                      f"classifier is Sequential={isinstance(model.model.classifier, nn.Sequential)}"
                      f"  -> l1-branch taken? {branch}")
                print(f"  l1 (B x Z)            : {tuple(l1.shape)}   embDim={embDim}")
                print(f"  outputs (B x C)       : {tuple(outputs.shape)}")

            if getattr(model, "multilabel", False):
                if use_true_labels:
                    targets = y.to(device).float()
                else:
                    targets = (torch.sigmoid(outputs) > 0.5).float()
                loss = F.binary_cross_entropy_with_logits(outputs, targets, reduction="sum")
                if trace: print(f"  [multilabel] targets  : {tuple(targets.shape)} {targets.dtype}"
                                f"  loss(BCE,sum)={loss.item():.3f}")
            else:
                if use_true_labels:
                    targets = y.to(device).long()
                else:
                    targets = torch.argmax(outputs, dim=1)
                loss = F.cross_entropy(outputs, targets, reduction="sum")
                if trace: print(f"  [softmax]   targets   : {tuple(targets.shape)} {targets.dtype}"
                                f"  loss(CE,sum)={loss.item():.3f}")

            l0_grads = torch.autograd.grad(loss, outputs)[0]     # B x C
            if trace: print(f"  l0_grads (B x C)      : {tuple(l0_grads.shape)}"
                            f"   = p_x - e_y  (softmax) / sigmoid - y (BCE)")

            if grad_embedding_type != "bias":
                l0_expand = torch.repeat_interleave(l0_grads, embDim, dim=1)   # B x C*Z
                l1_grads  = l0_expand * l1.repeat(1, outputs.shape[-1])        # B x C*Z
                if trace:
                    print(f"  l0_expand (B x C*Z)   : {tuple(l0_expand.shape)}   (each of C errors x{embDim})")
                    print(f"  l1.repeat (B x Z*C)   : {tuple(l1.repeat(1, outputs.shape[-1]).shape)}   (h_x tiled x{outputs.shape[-1]})")
                    print(f"  l1_grads (B x C*Z)    : {tuple(l1_grads.shape)}   = flattened (p-e_y) outer h_x")

            if grad_embedding_type == "bias":
                gradient_embedding = l0_grads
            elif grad_embedding_type == "linear":
                gradient_embedding = l1_grads
            else:
                gradient_embedding = torch.cat([l0_grads, l1_grads], dim=1)
            if i == 0:
                gradient_embeddings = torch.empty(
                    [len(dataloader.dataset), gradient_embedding.shape[1]], device="cpu")
                print(f"  gradient_embedding/row: {tuple(gradient_embedding.shape)}"
                      f"  -> allocate (N={len(dataloader.dataset)}, {gradient_embedding.shape[1]}) on CPU")
            gradient_embeddings[start_index:start_index + gradient_embedding.shape[0]] = \
                gradient_embedding.detach().to("cpu")
            start_index += gradient_embedding.shape[0]
            torch.cuda.empty_cache()
    print(f"  RETURN gradient_embeddings: {tuple(gradient_embeddings.shape)}  (N x C*Z)\n")
    return gradient_embeddings

### 6a. CIFAR-10 (single-label softmax, `small_head=True`)

In [ ]:
_ = get_grad_embedding_traced(model_cifar, loader_cifar, tag="CIFAR-10 ResNet")

### 6b. MIMIC-pheno (multi-label sigmoid/BCE, bare `Linear` head)

In [ ]:
_ = get_grad_embedding_traced(model_pheno, loader_pheno, tag="MIMIC-pheno SAND")

## 7. Things to play with

* **`grad_embedding_type`** — try `"bias"` (returns just `l0_grads`, shape `(N, C)`)
  or anything else (concatenation, `(N, C + C*Z)`).
* **`use_true_labels`** — `False` = BADGE pseudo-labels (argmax / sigmoid>0.5, the pool
  convention); `True` = ground-truth `y` (only meaningful on the labeled set).
* **The `l1`-branch (line 289 in source)** — neither example enters it. Compose a
  config with a non-small-head *Sequential* head to trigger it, e.g.
  `compose_cfg(["model=resnet","data=cifar10","model.small_head=False", ...])`
  and re-run the trace.
* **Verify the outer-product structure** — the cell below checks that block `c` of
  `l1_grads` equals `l0_grads[:, c] * l1`.

In [ ]:
# Reconstruct l1_grads by hand for CIFAR and compare to torch.repeat_interleave path.
with disable_mc_dropout(model_cifar):
    (x, y) = next(iter(loader_cifar))
    x = x.to(DEVICE)
    with torch.no_grad():
        feats = model_cifar.get_features(x)          # (B, Z)
    out = model_cifar.model.classifier(feats)        # (B, C)
    tgt = torch.argmax(out, dim=1)
    loss = F.cross_entropy(out, tgt, reduction="sum")
    g0 = torch.autograd.grad(loss, out)[0]           # (B, C)

B, C = out.shape
Z = feats.shape[-1]
manual = (g0.unsqueeze(2) * feats.unsqueeze(1)).reshape(B, C * Z)      # (B,C,Z)->(B,C*Z)
fast   = torch.repeat_interleave(g0, Z, dim=1) * feats.repeat(1, C)    # the source path
print("B,C,Z =", B, C, Z)
print("max abs diff (manual outer vs source path):",
      (manual - fast).abs().max().item())